# Anima x ComfyUI Colab (v11 - Prompt 编辑器版)

**关键**: Cell 9 改成可编辑 PROMPTS 列表, 直接在 cell 里改 prompt, 不用 wget, 不用碰 shell 引号。

**架构**:
- Cell 1-2: 装 ComfyUI + 下模型
- Cell 3-5: cloudflared + 启服务 + 健康检查
- Cell 6: 内嵌脚本和 workflow
- Cell 7: 单图测试
- Cell 8: 批量 5 张 (硬编码 seed)
- Cell 9: **Prompt 编辑器** - 改 PROMPTS 列表直接出图

In [ ]:
# Cell 1: 验证 GPU
import subprocess
try:
    out = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True)
    print('GPU:', out.strip())
except Exception as e:
    print('nvidia-smi 失败:', e)

In [ ]:
# Cell 2: 装 ComfyUI + 下载 Anima 三件套 (snapshot_download + shutil.copy)
import os, shutil
%cd /content
if not os.path.exists('ComfyUI'):
    get_ipython().system('git clone -q https://github.com/comfyanonymous/ComfyUI.git')
%cd /content/ComfyUI
get_ipython().system('pip install -q -r requirements.txt huggingface_hub hf_transfer')
get_ipython().system('mkdir -p models/diffusion_models models/text_encoders models/vae')
from huggingface_hub import snapshot_download
DOWNLOAD_DIR = '/content/Anima_download'
REPO = 'circlestone-labs/Anima'
snapshot_download(repo_id=REPO,
    allow_patterns=['split_files/diffusion_models/anima-preview.safetensors',
                    'split_files/text_encoders/qwen_3_06b_base.safetensors',
                    'split_files/vae/qwen_image_vae.safetensors'],
    local_dir=DOWNLOAD_DIR)
shutil.copy(f'{DOWNLOAD_DIR}/split_files/diffusion_models/anima-preview.safetensors','/content/ComfyUI/models/diffusion_models/anima-preview.safetensors')
shutil.copy(f'{DOWNLOAD_DIR}/split_files/text_encoders/qwen_3_06b_base.safetensors','/content/ComfyUI/models/text_encoders/qwen_3_06b_base.safetensors')
shutil.copy(f'{DOWNLOAD_DIR}/split_files/vae/qwen_image_vae.safetensors','/content/ComfyUI/models/vae/qwen_image_vae.safetensors')
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
print('Anima 三件套 ready')

In [ ]:
# Cell 3: 装 cloudflared
get_ipython().system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
get_ipython().system('cloudflared --version')

In [ ]:
# Cell 4: 启 ComfyUI + cloudflared
import subprocess, time, re, os
get_ipython().system('pkill -9 -f "python3 main.py" 2>/dev/null; pkill -9 -f cloudflared 2>/dev/null; sleep 1')
comfy = subprocess.Popen(['python3', 'main.py', '--listen', '0.0.0.0', '--port', '8188', '--disable-auto-launch'],cwd='/content/ComfyUI', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
started = False
for i in range(120):
    line = comfy.stdout.readline()
    if not line: time.sleep(1); continue
    if 'Starting server' in line or 'To see the GUI' in line:
        print('  [comfy]', line.strip()); started = True; break
cf = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8188', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
cf_url = None
for i in range(30):
    line = cf.stdout.readline()
    if not line: time.sleep(1); continue
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: cf_url = m.group(0); break
os.environ['COMFY_HOST'] = 'http://127.0.0.1:8188'
print('='*60)
print('ComfyUI API:', 'http://127.0.0.1:8188')
if cf_url: print('Browser UI:', cf_url)
print('='*60)

In [ ]:
# Cell 5: 健康检查 + 模型可见性
import urllib.request, json, os
host = os.environ.get('COMFY_HOST', 'http://127.0.0.1:8188')
try:
    with urllib.request.urlopen(f'{host}/system_stats', timeout=15) as r:
        d = json.loads(r.read())
    print(f"ComfyUI {d.get('system', {}).get('comfyui_version', '?')} OK")
    with urllib.request.urlopen(f'{host}/models', timeout=15) as r2:
        models = json.loads(r2.read())
    print('text_encoders:', models.get('text_encoders', []))
    print('diffusion_models:', models.get('diffusion_models', []))
    print('vae:', models.get('vae', []))
except Exception as e:
    print(f'无响应: {e}, 等几秒重跑')

In [ ]:
# Cell 6: 内嵌脚本和 workflow
import base64, os, json as _json
PY_B64 = 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKQW5pbWEgeCBDb21meVVJIOeoi+W6j+WMlueUn+aIkOiEmuacrCAodjEwIC0gc3Rkb3V0IOmUmeivr+aJk+WNsCArIOWuouaIt+err+mihOagoemqjCkKIiIiCmltcG9ydCBhcmdwYXJzZSwganNvbiwgdXVpZCwgdXJsbGliLnJlcXVlc3QsIHVybGxpYi5wYXJzZSwgdGltZSwgb3MsIHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCkNPTUZZX0hPU1QgPSBvcy5lbnZpcm9uLmdldCgiQ09NRllfSE9TVCIsICJodHRwOi8vMTI3LjAuMC4xOjgxODgiKQpXT1JLRkxPV19QQVRIID0gUGF0aChfX2ZpbGVfXykucGFyZW50IC8gImFuaW1hX3dvcmtmbG93Lmpzb24iCgpERUZBVUxUX1BST01QVCA9ICgKICAgICJtYXN0ZXJwaWVjZSwgYmVzdCBxdWFsaXR5LCBzY29yZV83LCBzYWZlLiAiCiAgICAiQW4gYW5pbWUgZ2lybCB3ZWFyaW5nIGEgYmxhY2sgdGFuay10b3AgYW5kIGRlbmltIHNob3J0cyBzdGFuZGluZyBvdXRkb29ycywgIgogICAgImhvbGRpbmcgYSByZWN0YW5ndWxhciBzaWduIHJlYWRpbmcgJ0FOSU1BJywgc21pbGluZyBhdCB2aWV3ZXIsICIKICAgICJ0cmVlcyBhbmQgYmx1ZSBza3kgd2l0aCBjbG91ZHMgaW4gYmFja2dyb3VuZCIKKQpERUZBVUxUX05FR0FUSVZFID0gIndvcnN0IHF1YWxpdHksIGxvdyBxdWFsaXR5LCBzY29yZV8xLCBzY29yZV8yLCBzY29yZV8zLCBibHVycnksIGpwZWcgYXJ0aWZhY3RzLCBzZXBpYSIKCgpkZWYgcHJlY2hlY2tfd29ya2Zsb3cod29ya2Zsb3c6IGRpY3QpOgogICAgIiIi6LCDIC9vYmplY3RfaW5mbyDpqozor4EgTG9hZGVyIOiKgueCueiDveaJvuWIsOaIkeS7rOeahOaooeWeiyArIOaJgOacieWPguaVsOWQiOazlSIiIgogICAgcHJpbnQoZiJbcHJlY2hlY2tdIHF1ZXJ5aW5nIHtDT01GWV9IT1NUfS9vYmplY3RfaW5mbyAuLi4iKQogICAgdHJ5OgogICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3BlbihmIntDT01GWV9IT1NUfS9vYmplY3RfaW5mbyIsIHRpbWVvdXQ9MTUpIGFzIHI6CiAgICAgICAgICAgIGluZm8gPSBqc29uLmxvYWRzKHIucmVhZCgpKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHByaW50KGYiW3ByZWNoZWNrXSBGQUlMOiBjYW5ub3QgcmVhY2ggL29iamVjdF9pbmZvOiB7ZX0iKQogICAgICAgIHJldHVybiBGYWxzZQoKICAgIG9rID0gVHJ1ZQogICAgZm9yIG5pZCwgbm9kZSBpbiB3b3JrZmxvdy5pdGVtcygpOgogICAgICAgIGN0ID0gbm9kZS5nZXQoImNsYXNzX3R5cGUiKQogICAgICAgIGlucHV0cyA9IG5vZGUuZ2V0KCJpbnB1dHMiLCB7fSkKICAgICAgICBpZiBjdCBub3QgaW4gaW5mbzoKICAgICAgICAgICAgcHJpbnQoZiJbcHJlY2hlY2tdIG5vZGUge25pZH0gY2xhc3MgJ3tjdH0nIG5vdCBmb3VuZCBvbiBzZXJ2ZXIiKQogICAgICAgICAgICBvayA9IEZhbHNlCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbm9kZV9pbmZvID0gaW5mb1tjdF0uZ2V0KCJpbnB1dCIsIHt9KQogICAgICAgIHJlcXVpcmVkID0gbm9kZV9pbmZvLmdldCgicmVxdWlyZWQiLCB7fSkKICAgICAgICAjIOajgOafpeavj+S4qiBpbnB1dCDlrZfmrrXmmK/lkKblnKggc2VydmVyIOWumuS5iemHjAogICAgICAgIGZvciBrLCB2IGluIGlucHV0cy5pdGVtcygpOgogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHYsIGxpc3QpOgogICAgICAgICAgICAgICAgY29udGludWUgICMg6IqC54K55byV55SoCiAgICAgICAgICAgIGlmIGsgbm90IGluIHJlcXVpcmVkIGFuZCBrIG5vdCBpbiBub2RlX2luZm8uZ2V0KCJvcHRpb25hbCIsIHt9KToKICAgICAgICAgICAgICAgIHByaW50KGYiW3ByZWNoZWNrXSBub2RlIHtuaWR9IHtjdH0ue2t9PXt2IXJ9IG5vdCBpbiBzZXJ2ZXIgSU5QVVRfVFlQRVMiKQogICAgICAgICAgICAgICAgb2sgPSBGYWxzZQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZmllbGRfc3BlYyA9IHJlcXVpcmVkLmdldChrKSBvciBub2RlX2luZm8uZ2V0KCJvcHRpb25hbCIsIHt9KS5nZXQoaykKICAgICAgICAgICAgaWYgbm90IGZpZWxkX3NwZWM6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBvcHRpb25zID0gZmllbGRfc3BlY1swXSBpZiBpc2luc3RhbmNlKGZpZWxkX3NwZWMsIGxpc3QpIGVsc2UgTm9uZQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKG9wdGlvbnMsIGxpc3QpIGFuZCB2IG5vdCBpbiBvcHRpb25zOgogICAgICAgICAgICAgICAgcHJpbnQoZiJbcHJlY2hlY2tdIG5vZGUge25pZH0ge2N0fS57a309e3Yhcn0gTk9UIElOIHtvcHRpb25zfSIpCiAgICAgICAgICAgICAgICBvayA9IEZhbHNlCiAgICAgICAgICAgIGVsaWYgaXNpbnN0YW5jZShvcHRpb25zLCBkaWN0KSBhbmQgInJlbW90ZSIgaW4gb3B0aW9uczoKICAgICAgICAgICAgICAgICMgZm9sZGVyX3BhdGhzIOexu+Weiyzpqozor4Hmlofku7blrZjlnKgKICAgICAgICAgICAgICAgIHJlbW90ZV9saXN0ID0gb3B0aW9uc1sicmVtb3RlIl0KICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UocmVtb3RlX2xpc3QsIGxpc3QpIGFuZCB2IG5vdCBpbiByZW1vdGVfbGlzdDoKICAgICAgICAgICAgICAgICAgICBwcmludChmIltwcmVjaGVja10gbm9kZSB7bmlkfSB7Y3R9LntrfT17diFyfSBmaWxlIE5PVCBGT1VORCBvbiBzZXJ2ZXIiKQogICAgICAgICAgICAgICAgICAgIHByaW50KGYiW3ByZWNoZWNrXSAgIGF2YWlsYWJsZSBmaWxlczoge3JlbW90ZV9saXN0fSIpCiAgICAgICAgICAgICAgICAgICAgb2sgPSBGYWxzZQogICAgaWYgb2s6CiAgICAgICAgcHJpbnQoIltwcmVjaGVja10gT0s6IGFsbCBpbnB1dHMgdmFsaWQiKQogICAgZWxzZToKICAgICAgICBwcmludCgiW3ByZWNoZWNrXSBGQUlMRUQgLSBmaXggYWJvdmUgaXNzdWVzIGJlZm9yZSBzdWJtaXR0aW5nIikKICAgIHJldHVybiBvawoKCmRlZiBxdWV1ZV9wcm9tcHQod29ya2Zsb3c6IGRpY3QpIC0+IHN0cjoKICAgIGNsaWVudF9pZCA9IHN0cih1dWlkLnV1aWQ0KCkpCiAgICBwYXlsb2FkID0geyJwcm9tcHQiOiB3b3JrZmxvdywgImNsaWVudF9pZCI6IGNsaWVudF9pZH0KICAgIGRhdGEgPSBqc29uLmR1bXBzKHBheWxvYWQpLmVuY29kZSgidXRmLTgiKQogICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCgKICAgICAgICBmIntDT01GWV9IT1NUfS9wcm9tcHQiLAogICAgICAgIGRhdGE9ZGF0YSwKICAgICAgICBoZWFkZXJzPXsiQ29udGVudC1UeXBlIjogImFwcGxpY2F0aW9uL2pzb24ifQogICAgKQogICAgdHJ5OgogICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3BlbihyZXEsIHRpbWVvdXQ9NjApIGFzIHI6CiAgICAgICAgICAgIHJlc3VsdCA9IGpzb24ubG9hZHMoci5yZWFkKCkpCiAgICBleGNlcHQgdXJsbGliLmVycm9yLkhUVFBFcnJvciBhcyBlOgogICAgICAgIGJvZHkgPSBlLnJlYWQoKS5kZWNvZGUoInV0Zi04IiwgZXJyb3JzPSJyZXBsYWNlIikKICAgICAgICBwcmludChmIlxuPT09IFtxdWV1ZV9wcm9tcHRdIEhUVFAge2UuY29kZX0ge2UucmVhc29ufSA9PT0iKQogICAgICAgIHByaW50KGYiPT09IFtxdWV1ZV9wcm9tcHRdIFJlc3BvbnNlIGJvZHk6ID09PVxue2JvZHl9XG4iKQogICAgICAgICMg5oqK5omA5pyJIExvYWRlciDoioLngrnlj4LmlbDmiZPlh7rmnaXmlrnkvr/lr7nnhacKICAgICAgICBmb3IgbmlkLCBub2RlIGluIHdvcmtmbG93Lml0ZW1zKCk6CiAgICAgICAgICAgIGN0ID0gbm9kZS5nZXQoImNsYXNzX3R5cGUiLCAiPyIpCiAgICAgICAgICAgIGlmICJMb2FkZXIiIGluIGN0IG9yICJLU2FtcGxlciIgaW4gY3Q6CiAgICAgICAgICAgICAgICBwcmludChmIiAgIHNlbnQ6IG5vZGUge25pZH0ge2N0fToge25vZGUuZ2V0KCdpbnB1dHMnLCB7fSl9IikKICAgICAgICByYWlzZQogICAgZXhjZXB0IHVybGxpYi5lcnJvci5VUkxFcnJvciBhcyBlOgogICAgICAgIHByaW50KGYiW3F1ZXVlX3Byb21wdF0gVVJMRXJyb3I6IHtlLnJlYXNvbn0iKQogICAgICAgIHJhaXNlCiAgICByZXR1cm4gcmVzdWx0WyJwcm9tcHRfaWQiXSwgY2xpZW50X2lkCgoKZGVmIHBvbGxfaGlzdG9yeShwcm9tcHRfaWQ6IHN0ciwgdGltZW91dF9zZWM6IGludCA9IDYwMCkgLT4gZGljdDoKICAgIGRlYWRsaW5lID0gdGltZS50aW1lKCkgKyB0aW1lb3V0X3NlYwogICAgbGFzdF9sb2cgPSAwCiAgICB3aGlsZSB0aW1lLnRpbWUoKSA8IGRlYWRsaW5lOgogICAgICAgIHRyeToKICAgICAgICAgICAgd2l0aCB1cmxsaWIucmVxdWVzdC51cmxvcGVuKGYie0NPTUZZX0hPU1R9L2hpc3Rvcnkve3Byb21wdF9pZH0iLCB0aW1lb3V0PTEwKSBhcyByOgogICAgICAgICAgICAgICAgaGlzdCA9IGpzb24ubG9hZHMoci5yZWFkKCkpCiAgICAgICAgICAgIGlmIHByb21wdF9pZCBpbiBoaXN0OgogICAgICAgICAgICAgICAgcmV0dXJuIGhpc3RbcHJvbXB0X2lkXQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgICAgICBub3cgPSB0aW1lLnRpbWUoKQogICAgICAgIGlmIG5vdyAtIGxhc3RfbG9nID49IDEwOgogICAgICAgICAgICBlbGFwc2VkID0gaW50KG5vdyAtIChkZWFkbGluZSAtIHRpbWVvdXRfc2VjKSkKICAgICAgICAgICAgcHJpbnQoZiIgIFtwb2xsXSB7ZWxhcHNlZH1zIGVsYXBzZWQsIHdhaXRpbmcuLi4iKQogICAgICAgICAgICBsYXN0X2xvZyA9IG5vdwogICAgICAgIHRpbWUuc2xlZXAoMS41KQogICAgcmFpc2UgVGltZW91dEVycm9yKGYiR2VuZXJhdGlvbiBub3QgZG9uZSBhZnRlciB7dGltZW91dF9zZWN9cyIpCgoKZGVmIGZldGNoX2ltYWdlKGZpbGVuYW1lOiBzdHIsIHN1YmZvbGRlcjogc3RyID0gIiIsIGZvbGRlcl90eXBlOiBzdHIgPSAib3V0cHV0IikgLT4gYnl0ZXM6CiAgICBxcyA9IHVybGxpYi5wYXJzZS51cmxlbmNvZGUoewogICAgICAgICJmaWxlbmFtZSI6IGZpbGVuYW1lLCAic3ViZm9sZGVyIjogc3ViZm9sZGVyLCAidHlwZSI6IGZvbGRlcl90eXBlCiAgICB9KQogICAgd2l0aCB1cmxsaWIucmVxdWVzdC51cmxvcGVuKGYie0NPTUZZX0hPU1R9L3ZpZXc/e3FzfSIsIHRpbWVvdXQ9MzApIGFzIHI6CiAgICAgICAgcmV0dXJuIHIucmVhZCgpCgoKZGVmIGdlbmVyYXRlKHByb21wdDogc3RyLCBuZWdhdGl2ZTogc3RyID0gREVGQVVMVF9ORUdBVElWRSwKICAgICAgICAgICAgIHdpZHRoOiBpbnQgPSAxMDI0LCBoZWlnaHQ6IGludCA9IDEwMjQsCiAgICAgICAgICAgICBzdGVwczogaW50ID0gMzAsIGNmZzogZmxvYXQgPSA0LjAsCiAgICAgICAgICAgICBzYW1wbGVyOiBzdHIgPSAiZXJfc2RlIiwgc2NoZWR1bGVyOiBzdHIgPSAic2ltcGxlIiwKICAgICAgICAgICAgIHNlZWQ6IGludCA9IC0xLAogICAgICAgICAgICAgd29ya2Zsb3dfcGF0aDogc3RyID0gTm9uZSwgc2tpcF9wcmVjaGVjazogYm9vbCA9IEZhbHNlKSAtPiBieXRlczoKICAgIHdmX3BhdGggPSBQYXRoKHdvcmtmbG93X3BhdGgpIGlmIHdvcmtmbG93X3BhdGggZWxzZSBXT1JLRkxPV19QQVRICiAgICB3ZiA9IGpzb24ubG9hZHMod2ZfcGF0aC5yZWFkX3RleHQoKSkKCiAgICB3ZlsiMTEiXVsiaW5wdXRzIl1bInRleHQiXSA9IHByb21wdAogICAgd2ZbIjEyIl1bImlucHV0cyJdWyJ0ZXh0Il0gPSBuZWdhdGl2ZQogICAgd2ZbIjI4Il1bImlucHV0cyJdWyJ3aWR0aCJdID0gd2lkdGgKICAgIHdmWyIyOCJdWyJpbnB1dHMiXVsiaGVpZ2h0Il0gPSBoZWlnaHQKICAgIHdmWyIxOSJdWyJpbnB1dHMiXVsic3RlcHMiXSA9IHN0ZXBzCiAgICB3ZlsiMTkiXVsiaW5wdXRzIl1bImNmZyJdID0gY2ZnCiAgICB3ZlsiMTkiXVsiaW5wdXRzIl1bInNhbXBsZXJfbmFtZSJdID0gc2FtcGxlcgogICAgd2ZbIjE5Il1bImlucHV0cyJdWyJzY2hlZHVsZXIiXSA9IHNjaGVkdWxlcgogICAgd2ZbIjE5Il1bImlucHV0cyJdWyJzZWVkIl0gPSBzZWVkIGlmIHNlZWQgPj0gMCBlbHNlIGludC5mcm9tX2J5dGVzKG9zLnVyYW5kb20oOCksICJiaWciKQoKICAgIGlmIG5vdCBza2lwX3ByZWNoZWNrOgogICAgICAgIGlmIG5vdCBwcmVjaGVja193b3JrZmxvdyh3Zik6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigicHJlY2hlY2sgZmFpbGVkIC0gc2VlIGFib3ZlIikKCiAgICBwcmludChmIltnZW5dIGhvc3Q9e0NPTUZZX0hPU1R9IHN0ZXBzPXtzdGVwc30ge3dpZHRofXh7aGVpZ2h0fSBzYW1wbGVyPXtzYW1wbGVyfSIpCiAgICBwaWQsIF8gPSBxdWV1ZV9wcm9tcHQod2YpCiAgICBwcmludChmIltnZW5dIHByb21wdF9pZCA9IHtwaWR9LCB3YWl0aW5nLi4uIikKCiAgICBoaXN0ID0gcG9sbF9oaXN0b3J5KHBpZCwgdGltZW91dF9zZWM9c3RlcHMgKiAzMCArIDYwKQogICAgb3V0cHV0cyA9IGhpc3QuZ2V0KCJvdXRwdXRzIiwge30pCiAgICBpbWdfaW5mbyA9IG91dHB1dHMuZ2V0KCIxIiwge30pLmdldCgiaW1hZ2VzIiwgW3t9XSlbMF0KICAgIGZpbGVuYW1lID0gaW1nX2luZm9bImZpbGVuYW1lIl0KICAgIHByaW50KGYiW2dlbl0gZG9uZSwgZmV0Y2hpbmcge2ZpbGVuYW1lfSIpCiAgICByZXR1cm4gZmV0Y2hfaW1hZ2UoZmlsZW5hbWUsIHN1YmZvbGRlcj1pbWdfaW5mby5nZXQoInN1YmZvbGRlciIsICIiKSwKICAgICAgICAgICAgICAgICAgICAgICBmb2xkZXJfdHlwZT1pbWdfaW5mby5nZXQoInR5cGUiLCAib3V0cHV0IikpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCJwcm9tcHQiLCBuYXJncz0iPyIsIGRlZmF1bHQ9REVGQVVMVF9QUk9NUFQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi1uIiwgIi0tbmVnYXRpdmUiLCBkZWZhdWx0PURFRkFVTFRfTkVHQVRJVkUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi1XIiwgIi0td2lkdGgiLCB0eXBlPWludCwgZGVmYXVsdD0xMDI0KQogICAgYXAuYWRkX2FyZ3VtZW50KCItSCIsICItLWhlaWdodCIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMjQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi1zIiwgIi0tc3RlcHMiLCB0eXBlPWludCwgZGVmYXVsdD0zMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLWMiLCAiLS1jZmciLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTQuMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zYW1wbGVyIiwgZGVmYXVsdD0iZXJfc2RlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zY2hlZHVsZXIiLCBkZWZhdWx0PSJzaW1wbGUiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD0tMSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLW8iLCAiLS1vdXRwdXQiLCBkZWZhdWx0PSJhbmltYV9vdXRwdXQucG5nIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1ob3N0IiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXdvcmtmbG93IiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNraXAtcHJlY2hlY2siLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGdsb2JhbCBDT01GWV9IT1NUCiAgICBpZiBhcmdzLmhvc3Q6CiAgICAgICAgQ09NRllfSE9TVCA9IGFyZ3MuaG9zdAoKICAgIHBuZyA9IGdlbmVyYXRlKAogICAgICAgIHByb21wdD1hcmdzLnByb21wdCwgbmVnYXRpdmU9YXJncy5uZWdhdGl2ZSwKICAgICAgICB3aWR0aD1hcmdzLndpZHRoLCBoZWlnaHQ9YXJncy5oZWlnaHQsCiAgICAgICAgc3RlcHM9YXJncy5zdGVwcywgY2ZnPWFyZ3MuY2ZnLAogICAgICAgIHNhbXBsZXI9YXJncy5zYW1wbGVyLCBzY2hlZHVsZXI9YXJncy5zY2hlZHVsZXIsCiAgICAgICAgc2VlZD1hcmdzLnNlZWQsIHdvcmtmbG93X3BhdGg9YXJncy53b3JrZmxvdywKICAgICAgICBza2lwX3ByZWNoZWNrPWFyZ3Muc2tpcF9wcmVjaGVjawogICAgKQogICAgUGF0aChhcmdzLm91dHB1dCkud3JpdGVfYnl0ZXMocG5nKQogICAgcHJpbnQoZiJbZ2VuXSBzYXZlZCB7YXJncy5vdXRwdXR9ICh7bGVuKHBuZyl9IGJ5dGVzKSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='
with open('/content/anima_gen.py', 'wb') as f:
    f.write(base64.b64decode(PY_B64))
JSON_B64 = 'ewogICIxIjogewogICAgImlucHV0cyI6IHsKICAgICAgImltYWdlcyI6IFsKICAgICAgICAiOCIsCiAgICAgICAgMAogICAgICBdCiAgICB9LAogICAgImNsYXNzX3R5cGUiOiAiUHJldmlld0ltYWdlIiwKICAgICJfbWV0YSI6IHsKICAgICAgInRpdGxlIjogIlByZXZpZXcgSW1hZ2UiCiAgICB9CiAgfSwKICAiOCI6IHsKICAgICJpbnB1dHMiOiB7CiAgICAgICJzYW1wbGVzIjogWwogICAgICAgICIxOSIsCiAgICAgICAgMAogICAgICBdLAogICAgICAidmFlIjogWwogICAgICAgICIxNSIsCiAgICAgICAgMAogICAgICBdCiAgICB9LAogICAgImNsYXNzX3R5cGUiOiAiVkFFRGVjb2RlIiwKICAgICJfbWV0YSI6IHsKICAgICAgInRpdGxlIjogIlZBRSBEZWNvZGUiCiAgICB9CiAgfSwKICAiMTEiOiB7CiAgICAiaW5wdXRzIjogewogICAgICAidGV4dCI6ICJtYXN0ZXJwaWVjZSwgYmVzdCBxdWFsaXR5LCBzY29yZV83LCBzYWZlLiBBbiBhbmltZSBnaXJsIHdlYXJpbmcgYSBibGFjayB0YW5rLXRvcCBhbmQgZGVuaW0gc2hvcnRzIGlzIHN0YW5kaW5nIG91dGRvb3JzLiBTaGUncyBob2xkaW5nIGEgcmVjdGFuZ3VsYXIgc2lnbiBvdXQgaW4gZnJvbnQgb2YgaGVyIHRoYXQgcmVhZHMgXCJBTklNQVwiLiBTaGUncyBsb29raW5nIGF0IHRoZSB2aWV3ZXIgd2l0aCBhIHNtaWxlLiBUaGUgYmFja2dyb3VuZCBmZWF0dXJlcyBzb21lIHRyZWVzIGFuZCBibHVlIHNreSB3aXRoIGNsb3Vkcy4iLAogICAgICAiY2xpcCI6IFsKICAgICAgICAiNDUiLAogICAgICAgIDAKICAgICAgXQogICAgfSwKICAgICJjbGFzc190eXBlIjogIkNMSVBUZXh0RW5jb2RlIiwKICAgICJfbWV0YSI6IHsKICAgICAgInRpdGxlIjogIkNMSVAgVGV4dCBFbmNvZGUgKFBvc2l0aXZlIFByb21wdCkiCiAgICB9CiAgfSwKICAiMTIiOiB7CiAgICAiaW5wdXRzIjogewogICAgICAidGV4dCI6ICJ3b3JzdCBxdWFsaXR5LCBsb3cgcXVhbGl0eSwgc2NvcmVfMSwgc2NvcmVfMiwgc2NvcmVfMywgYmx1cnJ5LCBqcGVnIGFydGlmYWN0cywgc2VwaWEiLAogICAgICAiY2xpcCI6IFsKICAgICAgICAiNDUiLAogICAgICAgIDAKICAgICAgXQogICAgfSwKICAgICJjbGFzc190eXBlIjogIkNMSVBUZXh0RW5jb2RlIiwKICAgICJfbWV0YSI6IHsKICAgICAgInRpdGxlIjogIkNMSVAgVGV4dCBFbmNvZGUgKE5lZ2F0aXZlIFByb21wdCkiCiAgICB9CiAgfSwKICAiMTUiOiB7CiAgICAiaW5wdXRzIjogewogICAgICAidmFlX25hbWUiOiAicXdlbl9pbWFnZV92YWUuc2FmZXRlbnNvcnMiCiAgICB9LAogICAgImNsYXNzX3R5cGUiOiAiVkFFTG9hZGVyIiwKICAgICJfbWV0YSI6IHsKICAgICAgInRpdGxlIjogIkxvYWQgVkFFIgogICAgfQogIH0sCiAgIjE5IjogewogICAgImlucHV0cyI6IHsKICAgICAgInNlZWQiOiA4NTY4NTM2NTc1MzUxNDgsCiAgICAgICJzdGVwcyI6IDMwLAogICAgICAiY2ZnIjogNC4wLAogICAgICAic2FtcGxlcl9uYW1lIjogImVyX3NkZSIsCiAgICAgICJzY2hlZHVsZXIiOiAic2ltcGxlIiwKICAgICAgImRlbm9pc2UiOiAxLjAsCiAgICAgICJtb2RlbCI6IFsKICAgICAgICAiNDQiLAogICAgICAgIDAKICAgICAgXSwKICAgICAgInBvc2l0aXZlIjogWwogICAgICAgICIxMSIsCiAgICAgICAgMAogICAgICBdLAogICAgICAibmVnYXRpdmUiOiBbCiAgICAgICAgIjEyIiwKICAgICAgICAwCiAgICAgIF0sCiAgICAgICJsYXRlbnRfaW1hZ2UiOiBbCiAgICAgICAgIjI4IiwKICAgICAgICAwCiAgICAgIF0KICAgIH0sCiAgICAiY2xhc3NfdHlwZSI6ICJLU2FtcGxlciIsCiAgICAiX21ldGEiOiB7CiAgICAgICJ0aXRsZSI6ICJLU2FtcGxlciIKICAgIH0KICB9LAogICIyOCI6IHsKICAgICJpbnB1dHMiOiB7CiAgICAgICJ3aWR0aCI6IDEwMjQsCiAgICAgICJoZWlnaHQiOiAxMDI0LAogICAgICAiYmF0Y2hfc2l6ZSI6IDEKICAgIH0sCiAgICAiY2xhc3NfdHlwZSI6ICJFbXB0eUxhdGVudEltYWdlIiwKICAgICJfbWV0YSI6IHsKICAgICAgInRpdGxlIjogIkVtcHR5IExhdGVudCBJbWFnZSIKICAgIH0KICB9LAogICI0NCI6IHsKICAgICJpbnB1dHMiOiB7CiAgICAgICJ1bmV0X25hbWUiOiAiYW5pbWEtcHJldmlldy5zYWZldGVuc29ycyIsCiAgICAgICJ3ZWlnaHRfZHR5cGUiOiAiZGVmYXVsdCIKICAgIH0sCiAgICAiY2xhc3NfdHlwZSI6ICJVTkVUTG9hZGVyIiwKICAgICJfbWV0YSI6IHsKICAgICAgInRpdGxlIjogIkxvYWQgRGlmZnVzaW9uIE1vZGVsIgogICAgfQogIH0sCiAgIjQ1IjogewogICAgImlucHV0cyI6IHsKICAgICAgImNsaXBfbmFtZSI6ICJxd2VuXzNfMDZiX2Jhc2Uuc2FmZXRlbnNvcnMiLAogICAgICAidHlwZSI6ICJzdGFibGVfZGlmZnVzaW9uIiwKICAgICAgImRldmljZSI6ICJkZWZhdWx0IgogICAgfSwKICAgICJjbGFzc190eXBlIjogIkNMSVBMb2FkZXIiLAogICAgIl9tZXRhIjogewogICAgICAidGl0bGUiOiAiTG9hZCBDTElQIgogICAgfQogIH0KfQ=='
with open('/content/anima_workflow.json', 'wb') as f:
    f.write(base64.b64decode(JSON_B64))
wf = _json.load(open('/content/anima_workflow.json'))
print('脚本 + workflow 已就绪')

In [ ]:
# Cell 7: 单图测试
import os
os.environ['COMFY_HOST'] = 'http://127.0.0.1:8188'
%cd /content
get_ipython().system(
    "python3 /content/anima_gen.py "
    "'masterpiece, best quality, score_7, safe. 1girl, white hair, blue eyes, classroom window, sunlight, smile' "
    "-o /content/anima_test1.png -W 1024 -H 1024 -s 30 -c 4.0"
)
from IPython.display import Image, display
display(Image('/content/anima_test1.png'))

In [ ]:
# Cell 8: 批量 5 张 (硬编码 seed, 快速抽卡)
import os
os.environ['COMFY_HOST'] = 'http://127.0.0.1:8188'
%cd /content
PROMPT = 'masterpiece, best quality, score_7, safe. 1girl, long black hair, red eyes, shrine maiden, cherry blossoms, night sky'
NEG = 'worst quality, low quality, score_1, score_2, score_3, blurry, jpeg artifacts'
for i, seed in enumerate([12345, 67890, 11111, 22222, 33333]):
    out = f'/content/anima_batch_{i}.png'
    print(f'[{i+1}/5] seed={seed}')
    get_ipython().system(f"python3 /content/anima_gen.py '{PROMPT}' -n '{NEG}' -o {out} -W 1024 -H 1024 -s 30 -c 4.0 --seed {seed}")
from IPython.display import Image, display
for i in range(5):
    print(f'--- batch_{i} ---')
    display(Image(f'/content/anima_batch_{i}.png'))

In [ ]:
# Cell 9: Prompt 编辑器 + 循环生成
# 在下面 PROMPTS 列表里改/加条目, 每个 dict 一张图

# ============================================================
# 在这里改 prompts - 每张图一行 prompt + 一行 negative (可选)
# 格式: {"prompt": "...", "negative": "...", "seed": 12345, "steps": 30, "cfg": 4, "w": 1024, "h": 1024, "sampler": "er_sde"}
# ============================================================
PROMPTS = [
    {
        "prompt": "masterpiece, best quality, score_7, safe. 1girl, silver hair, golden eyes, ornate dress, magic circle, dark background",
        "negative": "worst quality, low quality, score_1, score_2, score_3, blurry, jpeg artifacts",
        "seed": -1,
        "steps": 40,
        "cfg": 4.5,
        "w": 768,
        "h": 1280,
        "sampler": "euler_ancestral",
    },
    # === 复制上面整块加新条目 ===
    # {
    #     "prompt": "1girl, cat ears, maid, kitchen",
    #     "negative": "low quality",
    #     "seed": 42,
    #     "steps": 30,
    #     "cfg": 4.0,
    #     "w": 1024,
    #     "h": 1024,
    #     "sampler": "er_sde",
    # },
]

import os, json, subprocess, shlex
from pathlib import Path

os.environ['COMFY_HOST'] = 'http://127.0.0.1:8188'
os.environ['COMFY_HOST'] = 'http://127.0.0.1:8188'

# === 健康检查: ComfyUI 必须先在 8188 起来, 没起就提示用户跑 Cell 4 ===
import urllib.request, urllib.error
try:
    with urllib.request.urlopen(f'{os.environ["COMFY_HOST"]}/system_stats', timeout=5) as r:
        v = json.loads(r.read()).get('system', {}).get('comfyui_version', '?')
        print(f'[health] ComfyUI {v} OK at {os.environ["COMFY_HOST"]}')
except (urllib.error.URLError, ConnectionError) as e:
    raise SystemExit(
        f'[health] FAIL: {e}\n'
        f'  → 请先跑 Cell 4 启服务, 等打印 PUBLIC URL 后再回这里\n'
        f'  → 验证: 在另一个 cell 跑 `curl -s {os.environ["COMFY_HOST"]}/system_stats`'
    )

%cd /content

Path('/content/prompts.json').write_text(json.dumps(PROMPTS, ensure_ascii=False, indent=2))
print(f'已写入 {len(PROMPTS)} 条 prompt 到 /content/prompts.json')

from IPython.display import Image, display

for idx, item in enumerate(PROMPTS):
    out = f'/content/anima_prompt_{idx:02d}.png'
    prompt = item['prompt']
    negative = item.get('negative', 'worst quality, low quality, score_1, score_2, score_3')
    seed = item.get('seed', -1)
    steps = item.get('steps', 30)
    cfg = item.get('cfg', 4.0)
    w = item.get('w', 1024)
    h = item.get('h', 1024)
    sampler = item.get('sampler', 'euler_ancestral')

    # 写 prompt/negative 到临时 json, 用 python -c 调脚本避免 shell quoting
    args_json = json.dumps({'prompt': prompt, 'negative': negative, 'seed': seed,
                            'steps': steps, 'cfg': cfg, 'width': w, 'height': h,
                            'sampler': sampler, 'output': out})
    Path('/content/_args.json').write_text(args_json)

    print(f'\n[{idx+1}/{len(PROMPTS)}] {out} ({w}x{h}, steps={steps}, cfg={cfg}, seed={seed}, sampler={sampler})')
    print(f'  prompt: {prompt[:100]}...' if len(prompt) > 100 else f'  prompt: {prompt}')

    # 调脚本, 参数全从 json 读, 彻底避开 shell quoting
    code = """
import json, sys
sys.path.insert(0, '/content')
from anima_gen import generate
args = json.load(open('/content/_args.json'))
png = generate(
    prompt=args['prompt'], negative=args['negative'],
    width=args['width'], height=args['height'],
    steps=args['steps'], cfg=args['cfg'],
    sampler=args['sampler'], scheduler='simple', seed=args['seed'])
open(args['output'], 'wb').write(png)
print(f'  saved {args["output"    {
        "prompt": "masterpiece, best quality, score_7_up. 1girl, blue hair, simple background, masterpiece, best quality",
        "negative": "worst quality, low quality, score_1, score_2, score_3, blurry",
        "seed": -1,
        "steps": 40,
        "cfg": 4.5,
        "w": 768,
        "h": 1280,
        "sampler": "euler_ancestral",
    },
]} ({len(png)} bytes)')
"""
    Path('/content/_run.py').write_text(code)
    get_ipython().system('python3 /content/_run.py')

    # 显示
    print(f'  --- {out} ---')
    display(Image(out))

print(f'\n全部 {len(PROMPTS)} 张生成完成')
print('下载: 在左侧文件浏览器右键 PNG -> 下载, 或运行:')
print('  from google.colab import files; files.download("/content/anima_prompt_00.png")')

In [ ]:
# Cell 10 (auto-injected by colab_runner.py): upload PNGs to Discord webhook
import glob, os, base64, requests as _rq
_WEBHOOK_B64 = b''
WEBHOOK = base64.b64decode(_WEBHOOK_B64).decode() if _WEBHOOK_B64 else ''
if not WEBHOOK:
    print('[upload] no webhook configured; falling back to files.download')
    from google.colab import files as _cf
    for p in sorted(glob.glob('/content/anima_prompt_*.png')):
        _cf.download(p)
else:
    pngs = sorted(glob.glob('/content/anima_prompt_*.png'))
    print(f'[upload] pushing {len(pngs)} PNGs to Discord webhook')
    for p in pngs:
        with open(p, 'rb') as f:
            r = _rq.post(WEBHOOK, files={'file': (os.path.basename(p), f, 'image/png')})
        print(f'[upload] {os.path.basename(p)} -> HTTP {r.status_code}')
print('[upload] DONE')
